# **LightGBM 오류가 생기는 곳**

전체 점수가 좋아도 특정 매장이나 상품군에서 반복적으로 빗나가면 운영자가 쓰기 어렵습니다. 이 노트북은 05의 **Valid 예측**에서 예측 거리·날짜·상품군·매장별 오류를 살펴보고, 단순 기준선보다 나쁜 곳을 찾습니다.

오류 표는 개선 대상을 찾는 자료입니다. 판매량과 판촉 여부의 차이를 원인이나 인과효과로 단정하지 않습니다.

## **환경 준비**

Colab에서는 Drive를 연결합니다. 로컬과 Colab 모두 저장소 루트에서 실행합니다. 저장소 갱신은 별도의 00 노트북에서 마친 뒤 진행합니다.

In [ ]:
from pathlib import Path
import os

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

REPO = Path.cwd()
REPO = REPO.parent if REPO.name == "notebooks" else REPO
assert (REPO / "pyproject.toml").is_file(), "저장소 루트에서 실행해 주세요."
os.chdir(REPO)

## **패키지 준비**

Seaborn과 나눔고딕 글꼴을 설치합니다.


In [ ]:
%pip install -q seaborn koreanize-matplotlib


## **Library**


In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import koreanize_matplotlib

## **Utils**


In [ ]:
# 개별 예측 행에 절대오차와 로그 오차를 추가합니다.
def add_errors(frame):
    frame = frame.copy()
    frame["residual"] = frame["target_sales"] - frame["prediction"]
    frame["absolute_error"] = frame["residual"].abs()
    frame["squared_log_error"] = (
        np.log1p(frame["target_sales"].clip(lower=0))
        - np.log1p(frame["prediction"].clip(lower=0))
    ) ** 2
    return frame

In [ ]:
# 같은 그룹의 예측 오류와 판매량을 집계합니다.
def group_scores(frame, column):
    return (
        frame.groupby(column, observed=True)
        .agg(
            rows=("target_sales", "size"),
            actual_mean=("target_sales", "mean"),
            prediction_mean=("prediction", "mean"),
            mae=("absolute_error", "mean"),
            mean_squared_log_error=("squared_log_error", "mean"),
        )
        .assign(rmsle=lambda x: np.sqrt(x["mean_squared_log_error"]))
        .drop(columns="mean_squared_log_error")
        .sort_values("rmsle", ascending=False)
    )

In [ ]:
# 큰 절대오차 사례를 확인할 열만 남깁니다.
def top_errors(frame, count=20):
    columns = ["target_date", "store_nbr", "family", "target_sales", "prediction", "residual", "absolute_error"]
    return frame.nlargest(count, "absolute_error")[columns]

## **실행 설정**


In [ ]:
sns.set_theme(style="whitegrid", font="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.max_columns", 50)

PROJECT_ROOT = REPO
RUN_NAME = "full_history_time_split_fixed_i600"
OUTPUT_PATH = REPO / "outputs" / RUN_NAME

In [ ]:
metrics_path = OUTPUT_PATH / "metrics_lightgbm.json"
predictions_path = OUTPUT_PATH / "validation_predictions_lightgbm.csv"
importance_path = OUTPUT_PATH / "feature_importance_lightgbm.csv"
metrics_path = metrics_path if metrics_path.exists() else OUTPUT_PATH / "metrics.json"
predictions_path = predictions_path if predictions_path.exists() else OUTPUT_PATH / "validation_predictions.csv"
importance_path = importance_path if importance_path.exists() else OUTPUT_PATH / "feature_importance.csv"

with metrics_path.open(encoding="utf-8") as file:
    metrics = json.load(file)

predictions = pd.read_csv(predictions_path, parse_dates=["date", "target_date"])
importance = pd.read_csv(importance_path)
predictions = add_errors(predictions)

metrics_view = pd.DataFrame([metrics]).T.rename(columns={0: "value"})
metrics_view


In [ ]:
predictions["previous_week_baseline"] = predictions["sales_same_weekday_last_week"]
predictions["four_week_baseline"] = predictions["sales_same_weekday_4w_mean"]
predictions["baseline_squared_log_error"] = (
    np.log1p(predictions["target_sales"].clip(lower=0))
    - np.log1p(predictions["four_week_baseline"].clip(lower=0))
) ** 2

## **실제값과 예측값**

고판매 구간의 영향을 줄여 전체 분포를 보기 위해 로그 축을 사용합니다.

In [ ]:
sample = predictions.sample(min(30_000, len(predictions)), random_state=42)
limit = max(sample["target_sales"].max(), sample["prediction"].max())
residual_sample = predictions["residual"].clip(
    predictions["residual"].quantile(0.01), predictions["residual"].quantile(0.99),
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=sample, x="target_sales", y="prediction", alpha=0.15, s=8, ax=axes[0])
axes[0].plot([0, limit], [0, limit], linestyle="--", color="black", linewidth=1)
sns.histplot(residual_sample, bins=70, ax=axes[1])
axes[1].axvline(0, linestyle="--", color="black", linewidth=1)
axes[0].set(xscale="symlog", yscale="symlog", title="실제 판매와 예측", xlabel="실제", ylabel="예측")
axes[1].set(title="오차 분포 (양 끝 1% 제외)", xlabel="실제 - 예측")
fig.tight_layout()
plt.show()

## **예측 거리별 오차**

In [ ]:
horizon_scores = group_scores(predictions, "forecast_horizon").sort_index()
horizon_plot = horizon_scores.reset_index()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.lineplot(data=horizon_plot, x="forecast_horizon", y="rmsle", marker="o", ax=axes[0])
sns.lineplot(data=horizon_plot, x="forecast_horizon", y="mae", marker="o", ax=axes[1])
axes[0].set(title="예측 거리별 RMSLE", xlabel="며칠 뒤", ylabel="RMSLE")
axes[1].set(title="예측 거리별 MAE", xlabel="며칠 뒤", ylabel="MAE")
fig.tight_layout()
plt.show()

## **날짜별 안정성**

In [ ]:
daily_scores = group_scores(predictions, "target_date").sort_index()
daily_view = daily_scores[["rows", "rmsle", "mae"]].head()
daily_view


## **상품군별 오차**

RMSLE는 저판매 상품군의 상대적 오차에 민감하고, MAE는 판매 규모가 큰 상품군의 절대 오차에 민감합니다. 두 지표를 함께 확인합니다.

In [ ]:
family_scores = group_scores(predictions, "family")
family_view = family_scores.head(15)
family_view


In [ ]:
worst_rmsle = family_scores.head(12).sort_values("rmsle").reset_index()
worst_mae = family_scores.nlargest(12, "mae").sort_values("mae").reset_index()
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.barplot(data=worst_rmsle, x="rmsle", y="family", ax=axes[0], color="steelblue")
sns.barplot(data=worst_mae, x="mae", y="family", ax=axes[1], color="darkorange")
axes[0].set(title="RMSLE가 큰 상품군", xlabel="RMSLE", ylabel="")
axes[1].set(title="MAE가 큰 상품군", xlabel="MAE", ylabel="")
fig.tight_layout()
plt.show()

## **상품군별 계절 기준선 대비 개선**

양수는 LightGBM이 7일 전 판매량 기준선보다 개선된 상품군이며, 음수는 기준선보다 나빠진 상품군입니다.

In [ ]:
family_baseline = predictions.groupby("family", observed=True).agg(
    model_msle=("squared_log_error", "mean"),
    baseline_msle=("baseline_squared_log_error", "mean"),
)
family_baseline["model_rmsle"] = np.sqrt(family_baseline["model_msle"])
family_baseline["baseline_rmsle"] = np.sqrt(family_baseline["baseline_msle"])
family_baseline["improvement_pct"] = (
    1 - family_baseline["model_rmsle"] / family_baseline["baseline_rmsle"]
) * 100
family_baseline = family_baseline.sort_values("improvement_pct")
family_baseline_view = family_baseline[["model_rmsle", "baseline_rmsle", "improvement_pct"]]
family_baseline_view


In [ ]:
family_gain = family_baseline.reset_index()
family_gain["개선 방향"] = np.where(family_gain["improvement_pct"].ge(0), "개선", "악화")
fig, ax = plt.subplots(figsize=(9, 8))
sns.barplot(data=family_gain, x="improvement_pct", y="family",
            hue="개선 방향", dodge=False, ax=ax)
ax.axvline(0, color="black", linewidth=1)
ax.set(title="7일 계절 기준선 대비 RMSLE 개선", xlabel="개선율 (%)", ylabel="")
fig.tight_layout()
plt.show()

## **점포별 오차**

In [ ]:
store_scores = group_scores(predictions, "store_nbr")
store_view = store_scores.head(15)
store_view


## **프로모션·수요 규모별 오차**

목표일의 실제 판촉 여부는 결과를 나누어 보는 데만 사용합니다. 예측 시점에 알고 있던 입력 변수로 해석하지 않습니다.

In [ ]:
predictions["promoted"] = predictions["target_onpromotion"].gt(0)
predictions["demand_band"] = pd.cut(
    predictions["target_sales"],
    bins=[-0.001, 0, 5, 20, 100, np.inf],
    labels=["zero", "1-5", "6-20", "21-100", "over-100"],
)
promotion_scores = group_scores(predictions, "promoted")
demand_scores = group_scores(predictions, "demand_band")
promotion_scores

In [ ]:
demand_scores

## **큰 절대오차 사례**

In [ ]:
largest_errors = top_errors(predictions)
largest_errors


## **변수 중요도**

연도·주차처럼 시간과 함께 움직이는 변수는 실제 영향과 시간 추세의 대리효과가 섞일 수 있습니다. 중요도만으로 인과관계를 주장하지 않고 후속 제거 실험으로 확인합니다. 유가는 일별 수요와의 관계를 명확히 설명하기 어려워 입력에서 제외했습니다.

In [ ]:
top_importance = importance.head(20).sort_values("importance")
fig, ax = plt.subplots(figsize=(9, 7))
sns.barplot(data=top_importance, x="importance", y="feature", ax=ax, color="steelblue")
ax.set(title="LightGBM 변수 중요도", xlabel="분할 중요도", ylabel="")
fig.tight_layout()
plt.show()

## **Permutation Importance**

별도 분석 결과 파일이 있을 때만 표시합니다. 각 변수를 무작위로 섞었을 때 검증 RMSLE가 얼마나 악화되는지 측정합니다. 값이 클수록 해당 변수가 예측 성능에 더 크게 기여한 것입니다. 변수의 인과효과나 영향 방향을 의미하지는 않습니다.

In [ ]:
permutation_path = OUTPUT_PATH / "permutation_importance.csv"
if permutation_path.is_file():
    permutation = pd.read_csv(permutation_path)
    top_permutation = permutation.head(20).sort_values("importance_mean")
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.barplot(data=top_permutation, x="importance_mean", y="feature", ax=ax, color="steelblue")
    ax.errorbar(top_permutation["importance_mean"], range(len(top_permutation)),
                xerr=top_permutation["importance_std"], fmt="none", color="black")
    ax.axvline(0, color="black", linewidth=1)
    ax.set(title="검증 표본의 Permutation Importance", xlabel="변수 섞기 후 RMSLE 증가", ylabel="")
    fig.tight_layout()
    plt.show()


## **다음 해석 단계**

- 상품군·매장별 오류를 [08 평가](08_model_evaluation_and_decision.ipynb)의 7일 합계와 함께 읽습니다.
- 과거 다른 날짜에도 기준선 대비 개선이 이어지는지 [09 평가](09_lightgbm_historical_backtest.ipynb)로 확인합니다.
- 판촉과 오류의 관계는 관찰 결과이며 판촉의 인과효과로 주장하지 않습니다.